In [ ]:
import ROOT
import numpy as np
import pandas as pd
from scipy.stats import f as f_dist
import math
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati
ROOT.gStyle.SetFitFormat(
    "4.2e"
)  # con 2 cifre decimali

In [ ]:
file_path = "Esperienza5.xlsx"

sheets = {
    'Dati',
    'Manuele',
    'Alessio',
    'Emanuele',
    'Michele'
}

def load_data(sheet_name):
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name, skiprows=1)
        theta = pd.to_numeric(df.iloc[:, 0], errors='coerce')
        
        theta_val = theta.values.astype(np.float64)
        
        return theta_val
    except Exception as e:
        print(f"Errore nel caricamento del foglio {sheet_name}: {e}")
        return None, None

In [ ]:
root_objects = []
def make_hist(nomi, title, filename):
    c = ROOT.TCanvas(f"c_{filename}", title, 800, 600)
    c.SetGrid()
    

    for name in nomi:
        if name in data:
            theta = data[name]

            gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(theta), max(theta), 3)
            gaussian_fit.SetParNames("const", "mu", "sigma")
            gaussian_fit.SetLineColor(ROOT.kRed + 1)
            gaussian_fit.SetLineWidth(3)

            hist = ROOT.TH1F("hist", f"{title}; θ_B; conteggi", 10, min(theta), max(theta))
            
            for val in theta:
                hist.Fill(val)
                leg = ROOT.TLegend(0.15, 0.70, 0.40, 0.85)
                leg.SetBorderSize(1)
                leg.SetFillStyle(1001)

        gaussian_fit.SetParameters(hist.GetMaximum(), hist.GetMean(), hist.GetRMS())
            
        hist.SetLineColor(ROOT.kAzure + 3)
        hist.SetLineWidth(2)
        hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
        hist.GetXaxis().SetTitle("θ_B")
        hist.GetYaxis().SetTitle("conteggi")
        hist.SetTitle(f"{title}")

        r = hist.Fit(gaussian_fit, "S")

        fitStatus = r.Status()
        print(f"Fit status: {fitStatus}")

        parameters = gaussian_fit.GetParameters()
        errors = gaussian_fit.GetParErrors()

        chi2 = gaussian_fit.GetChisquare()

        print("Parametri stimati:")
        print(f"const = {parameters[0]} ± {errors[0]}")
        print(f"#mu = {parameters[1]} ± {errors[1]}")
        print(f"#sigma = {parameters[2]} ± {errors[2]}")
        print(f"Chi2 = {chi2}")

        root_objects.append(hist)

    hist.Draw()

    leg2 = ROOT.TLegend(0, 0.30, 0.15, 0.39)
    leg2.AddEntry(hist, "Residui", "f")
    leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
    leg2.Draw()

    c.SaveAs(filename)


In [ ]:
data = {}
for key, s_name in sheets.items():
    theta = load_data(s_name)
    data[key] = (theta) 


In [ ]:
make_hist['Dati', 'Istogramma con tutti i dati', 'hist_totalData.pdf']
make_hist['Manuele', 'Istogramma con i dati di Manuele', 'histManuData.pdf']
make_hist['Alessio', 'Istogramma con i dati di Alessio', 'histAleData.pdf']
make_hist['Michele', 'Istogramma con i dati di Michele', 'histMichData.pdf']
make_hist['Emanuele', 'Istogramma con i dati di Emanuele', 'histStroiliData.pdf']
